# Notebook 16 — Teaching Export Workflow

Demonstrates the complete Development Method pipeline that powers the Phase 1 teaching artefacts:
information cutoff enforcement, ATA factor selection, ATU projection, ultimate estimation,
JSON export, and Excel workbook generation.

**Uses:** `ibnr_utils.teaching_export`, `ibnr_utils.development`, `ibnr_utils.triangle_utils`  
**Produces:** `data/output/teaching_phase1/teaching_export_<cutoff>.json`,
`teaching_development_method_<cutoff>.xlsx`

## Step 1 — Configure Python path

**Purpose:** Locate the project root and add `src/` to `sys.path`.  
**Uses:** `pathlib`, `sys.path`  
**Produces:** `project_root`, `src_path`

In [1]:
import pathlib, sys

_cwd = pathlib.Path.cwd()
project_root = _cwd
for _ in range(5):
    if (project_root / 'src' / 'ibnr_utils').exists() and (project_root / 'README.md').exists():
        break
    project_root = project_root.parent

src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f'project_root: {project_root}')
print(f'src_path    : {src_path}')

project_root: /mnt/data/Linux/Documents/IBNR_Utils
src_path    : /mnt/data/Linux/Documents/IBNR_Utils/src


## Step 2 — Imports and constants

**Purpose:** Import the modules used throughout this notebook.  
**Uses:** `ibnr_utils.teaching_export`, `ibnr_utils.development`, `ibnr_utils.triangle_utils`, `ibnr_utils.workflow`  
**Produces:** All module names available for subsequent cells.  
**How it works:** The `teaching_export` module is a thin orchestration layer over the existing
`development`, `triangle_utils`, and `workflow` modules. All actuarial calculations are delegated
to those existing modules; `teaching_export` only coordinates them and serialises the results.

In [2]:
import pandas as pd
import numpy as np

from ibnr_utils.teaching_export import (
    SCHEMA_VERSION,
    TeachingExportConfig,
    TeachingExportResult,
    run_teaching_export,
    export_to_json,
    export_to_excel,
)
from ibnr_utils.development import AverageSpec, ata_matrix, atu_matrix, ultimate_matrix
from ibnr_utils.triangle_utils import cut_triangle_as_of, cumulative_to_incremental
from ibnr_utils.workflow import create_triangles_from_database, find_project_root

# Source CSV — synthetic only, never data/input/
CSV_PATH = project_root / 'data' / 'output' / 'synthetic_monthly_multiconcept_database.csv'
OUT_DIR  = project_root / 'data' / 'output' / 'teaching_phase1'
CUTOFF   = '2022-12'

print(f'CSV  : {CSV_PATH}')
print(f'Outdir: {OUT_DIR}')
print(f'Cutoff: {CUTOFF}')
print(f'Schema version: {SCHEMA_VERSION}')

CSV  : /mnt/data/Linux/Documents/IBNR_Utils/data/output/synthetic_monthly_multiconcept_database.csv
Outdir: /mnt/data/Linux/Documents/IBNR_Utils/data/output/teaching_phase1
Cutoff: 2022-12
Schema version: 1.0


## Step 3 — Load the raw triangle

**Purpose:** Load the synthetic monthly database and pivot it to a cumulative triangle.  
**Uses:** `create_triangles_from_database`, `CSV_PATH`  
**Produces:** `raw_triangle` (120 × 120 DataFrame for Loss Incurred)  
**How it works:** `create_triangles_from_database` returns one wide cumulative triangle per
concept. Each row is an accident period; each column is a development age. The upper-right
region is NaN because those calendar periods are in the future relative to the latest observed
diagonal. The shape before cutoff is determined solely by the simulation parameters
(n_accident_months=120, n_dev_periods=120).

In [3]:
wf = create_triangles_from_database(CSV_PATH, basis='month', concepts=['Loss Incurred'],
                                    run_validations=False, print_validation=False)
raw_triangle = wf.triangles['Loss Incurred']
print(f'Raw triangle shape: {raw_triangle.shape}')
print(f'Accident periods  : {raw_triangle.index[0]} … {raw_triangle.index[-1]}')
print(f'Development cols  : {raw_triangle.columns[0]} … {raw_triangle.columns[-1]}')
# Count observed vs future cells
n_obs = raw_triangle.notna().sum().sum()
n_tot = raw_triangle.shape[0] * raw_triangle.shape[1]
print(f'Observed cells    : {n_obs:,} / {n_tot:,} ({100*n_obs/n_tot:.1f} %)')

Raw triangle shape: (120, 120)
Accident periods  : 2016-01 … 2025-12
Development cols  : dev_0 … dev_119
Observed cells    : 7,260 / 14,400 (50.4 %)


## Step 4 — Apply an information cutoff

**Purpose:** Simulate the view an actuary had at a specific valuation date by zeroing out
all calendar periods that postdate the cutoff.  
**Uses:** `cut_triangle_as_of(triangle, cutoff)`, `CUTOFF = '2022-12'`  
**Produces:** `cut` — triangle with only observed cells up to and including 2022-12  
**How it works:** `cut_triangle_as_of` sets cells where `accident_period + dev_age > cutoff`
to NaN. This is the mechanism that enforces information discipline: the actuary cannot use
data that had not yet emerged at the valuation date. Changing `CUTOFF` to an earlier date
reduces the triangle and will produce different (typically more uncertain) ultimate estimates.
This is the key input to backtesting: repeating the projection at each historical cutoff and
comparing projected ultimates to eventual actual development.

In [4]:
cut = cut_triangle_as_of(raw_triangle, CUTOFF)
print(f'Cut triangle shape: {cut.shape}')
print(f'Accident periods  : {cut.index[0]} … {cut.index[-1]}')
n_obs_cut = cut.notna().sum().sum()
n_tot_cut = cut.shape[0] * cut.shape[1]
print(f'Observed cells    : {n_obs_cut:,} / {n_tot_cut:,} ({100*n_obs_cut/n_tot_cut:.1f} %)')

# Check the staircase pattern in the last few accident periods
last3 = cut.iloc[-3:]
obs_counts = last3.notna().sum(axis=1)
print()
print('Observed columns per last 3 accident periods (staircase pattern):')
for ap, cnt in obs_counts.items():
    print(f'  {ap}: {cnt} dev columns')

Cut triangle shape: (84, 84)
Accident periods  : 2016-01 … 2022-12
Observed cells    : 3,570 / 7,056 (50.6 %)

Observed columns per last 3 accident periods (staircase pattern):
  2022-10: 3 dev columns
  2022-11: 2 dev columns
  2022-12: 1 dev columns


## Step 5 — Compute the ATA matrix

**Purpose:** Compute age-to-age factors for multiple averaging methods and select one.  
**Uses:** `ata_matrix(cut, average_specs, selected=selected_average)`  
**Produces:** `ata` — DataFrame with one row per average spec plus a Selected row  
**How it works:** The ATA matrix rows each represent a different way to summarise
the historical factor triangle:
- **VW All** (volume-weighted, all periods): the default. Larger accident periods dominate.
- **VW 24 / VW 12**: weight by volume but only look at the last 24 or 12 development years.
  Useful when recent development behavior has changed structurally.
- **S All / S 12**: simple (unweighted) average. Each accident period has equal weight
  regardless of its size. More volatile than volume-weighted for heterogeneous books.

The `selected` parameter copies one row as "Selected". The ATU matrix is built from that row.
No tail factor is used here — we assume all accident periods are fully developed by
the last development age.

In [5]:
average_specs = [
    AverageSpec(method='weighted',               label='VW All'),
    AverageSpec(method='weighted', latest_periods=24, label='VW 24'),
    AverageSpec(method='weighted', latest_periods=12, label='VW 12'),
    AverageSpec(method='simple',   latest_periods=12, label='S 12'),
    AverageSpec(method='simple',                  label='S All'),
]

ata = ata_matrix(cut, average_specs, selected='VW All')
print(f'ATA matrix shape: {ata.shape}  (rows = averages + Selected, cols = dev steps)')
print()
# Show the first 6 columns for all rows
print('ATA matrix — first 6 development steps:')
print(ata.iloc[:, :6].to_string(float_format='{:.5f}'.format))
print()
print(f'Selected row: {ata.index[-1]}')

ATA matrix shape: (7, 83)  (rows = averages + Selected, cols = dev steps)

ATA matrix — first 6 development steps:
                 dev_1   dev_2   dev_3   dev_4   dev_5   dev_6
average_type                                                  
VW All         1.13524 1.13174 1.12783 1.12372 1.11902 1.11411
VW 24          1.13580 1.13236 1.12850 1.12425 1.11936 1.11444
VW 12          1.13533 1.13211 1.12798 1.12380 1.11821 1.11286
S 12           1.13589 1.13281 1.12878 1.12450 1.11904 1.11338
S All          1.13576 1.13231 1.12846 1.12436 1.11973 1.11483
Prior Selected 1.00000 1.00000 1.00000 1.00000 1.00000 1.00000
Selected       1.13524 1.13174 1.12783 1.12372 1.11902 1.11411

Selected row: Selected


## Step 6 — Compute the ATU matrix

**Purpose:** Convert age-to-age factors to age-to-ultimate (cumulative tail) factors.  
**Uses:** `atu_matrix(ata, selected_row='Selected')`  
**Produces:** `atu` — DataFrame with ATU and % Developed rows  
**How it works:** ATU at development age d = product of all selected ATA factors from
age d to the last observed age. The "% Developed" row = 1 / ATU, giving the fraction of
ultimate already earned at each development age. An ATU of 1.05 at age d means the actuary
expects a further 5% development before ultimate. "% Developed" = 1/1.05 ≈ 95.2%.
No tail factor → the ATU for the last development age is exactly 1.0 and % Developed = 100%.

In [6]:
atu = atu_matrix(ata)
print(f'ATU matrix shape: {atu.shape}')
print()
print('ATU matrix — first 6 development ages:')
print(atu.iloc[:, :6].to_string(float_format='{:.5f}'.format))
print()
# Check last column
print('ATU matrix — last development age:')
print(atu.iloc[:, -1])

ATU matrix shape: (6, 83)

ATU matrix — first 6 development ages:
                     dev_1   dev_2   dev_3   dev_4   dev_5   dev_6
Selected ATA       1.13524 1.13174 1.12783 1.12372 1.11902 1.11411
Prior Selected ATA 1.00000 1.00000 1.00000 1.00000 1.00000 1.00000
ATU                6.95566 6.12703 5.41380 4.80017 4.27168 3.81734
Prior ATU          1.00000 1.00000 1.00000 1.00000 1.00000 1.00000
% Developed        0.14377 0.16321 0.18471 0.20833 0.23410 0.26196
Prior % Developed  1.00000 1.00000 1.00000 1.00000 1.00000 1.00000

ATU matrix — last development age:
Selected ATA          1.000004
Prior Selected ATA    1.000000
ATU                   1.000004
Prior ATU             1.000000
% Developed           0.999996
Prior % Developed     1.000000
Name: dev_83, dtype: float64


## Step 7 — Project ultimates and compute additional development

**Purpose:** Apply the selected ATU factors to each accident period's current diagonal value.  
**Uses:** `ultimate_matrix(cut, atu)`, then compute `additional_development`  
**Produces:** `ult` — ultimate table with Actual, ATU, % Developed, Ultimate, flags  
**How it works:**
- **Actual** = current_incurred = value at the latest observed development period for that accident period.
- **Ultimate** = Actual × ATU
- **Additional development** = Ultimate − Actual. This is the expected future development
  under the Development Method. It is **not** IBNR in general: IBNR requires Claims Reported
  and paid data; additional development is simply the ATU projection minus what is already on the diagonal.
- **Flags:** `atu_below_1` (unexpected for non-decreasing concepts), `ultimate_below_actual`
  (mathematical consequence of atu<1), `percent_developed_above_1` (% dev > 100%).

In [7]:
ult = ultimate_matrix(cut, atu)
ult['additional_development'] = ult['Ultimate'] - ult['Actual']

print('Ultimate table — last 5 accident periods:')
display_cols = ['actual_development_period', 'Actual', 'ATU', '% Developed', 'Ultimate', 'additional_development']
print(ult[display_cols].tail(5).to_string(float_format='{:,.2f}'.format))
print()
print('Diagnostic flags (expect all 0):')
print(f'  atu_below_1            : {ult["atu_below_1"].sum()}')
print(f'  percent_developed_above_1: {ult["percent_developed_above_1"].sum()}')
print(f'  ultimate_below_actual  : {ult["ultimate_below_actual"].sum()}')
print()
total_actual = ult['Actual'].sum()
total_ult    = ult['Ultimate'].sum()
total_add    = ult['additional_development'].sum()
print(f'Total actual incurred  : {total_actual:>15,.0f}')
print(f'Total ultimate         : {total_ult:>15,.0f}')
print(f'Total addl development : {total_add:>15,.0f}')

Ultimate table — last 5 accident periods:
                actual_development_period     Actual  ATU  % Developed     Ultimate  additional_development
accident_period                                                                                            
2022-08                             dev_4 120,307.07 4.27         0.23   513,913.30              393,606.23
2022-09                             dev_3 237,776.13 4.80         0.21 1,141,366.00              903,589.87
2022-10                             dev_2  92,580.04 5.41         0.18   501,209.59              408,629.54
2022-11                             dev_1  50,420.96 6.13         0.16   308,930.87              258,509.91
2022-12                             dev_0  97,464.53 6.96         0.14   677,929.99              580,465.46

Diagnostic flags (expect all 0):
  atu_below_1            : 0
  percent_developed_above_1: 0
  ultimate_below_actual  : 0

Total actual incurred  :      30,229,474
Total ultimate         :      36,934,

## Step 8 — Run the full export via `run_teaching_export`

**Purpose:** Execute the same pipeline in one call using the high-level API.  
**Uses:** `TeachingExportConfig`, `run_teaching_export`  
**Produces:** `result` — `TeachingExportResult` with all computed artefacts  
**How it works:** `TeachingExportConfig` bundles all parameters — cutoff, concept, average
specs, selected average, tail factor, run ID — into a single object. `run_teaching_export`
applies the same steps as cells 3–7 above and returns a `TeachingExportResult` that holds
all intermediate and final artefacts. Running through the API guarantees reproducibility:
the same config always produces the same result.

In [8]:
config = TeachingExportConfig(
    cutoff='2022-12',
    csv_path=CSV_PATH,
    concept='Loss Incurred',
    selected_average='VW All',
    tail_factor=None,
)

result = run_teaching_export(config)
print(f'Run ID           : {result.config.run_id}')
print(f'Generated at     : {result.generated_at}')
print(f'Triangle shape   : {result.triangle_cumulative.shape}')
print(f'ATA matrix shape : {result.ata.shape}')
print(f'ATU matrix shape : {result.atu.shape}')
print(f'Ultimate records : {len(result.ultimate_table)}')

# Verify reconciliation: max diff between run_teaching_export and manual calculation above
diff = (result.ultimate_table['Ultimate'] - ult['Ultimate']).abs().max()
print(f'\nMax diff vs manual calculation: {diff:.2e}  (should be 0)')

Run ID           : 2e32e596-6d90-4aea-ad4e-fb637f8b4696
Generated at     : 2026-10-14T19:10:54.500847+00:00
Triangle shape   : (84, 84)
ATA matrix shape : (7, 83)
ATU matrix shape : (6, 83)
Ultimate records : 84

Max diff vs manual calculation: 0.00e+00  (should be 0)


## Step 9 — Export to JSON

**Purpose:** Serialise the result to the versioned JSON schema (v1.0).  
**Uses:** `export_to_json(result, output_path)`, `SCHEMA_VERSION`  
**Produces:** `teaching_export_2022-12.json` in `OUT_DIR`  
**How it works:** The JSON schema is the contract between Python and the HTML viewer.
Key design choices:
- Missing/future cells are `null`, never `0` (preserves information about observability).
- No `NaN` or `Infinity` literals (JSON does not define them; they break parsers).
- `development_ages` are integers (not "dev_N" strings) for easy JS arithmetic.
- Schema version "1.0" allows the viewer to reject incompatible future schemas.
- One file = one run and one cutoff; comparing across cutoffs means loading multiple files.

In [9]:
json_path = OUT_DIR / f'teaching_export_{CUTOFF}.json'
export_to_json(result, json_path)

import json
doc = json.loads(json_path.read_text())
print(f'JSON file size  : {json_path.stat().st_size / 1024:.1f} KB')
print(f'Schema version  : {doc["schema_version"]}')
print(f'Run ID          : {doc["run_id"]}')
print(f'Generated at    : {doc["generated_at"]}')
print(f'Accident periods: {len(doc["triangle"]["accident_periods"])}')
print(f'Dev ages        : {len(doc["triangle"]["development_ages"])}')
print(f'Ultimate records: {len(doc["ultimate_records"])}')
print()
# Verify no NaN / Inf
text = json_path.read_text()
for bad in ('NaN','Infinity','-Infinity'):
    if bad in text:
        print(f'ERROR: {bad!r} found in JSON')
    else:
        print(f'No {bad!r} in JSON — OK')

JSON file size  : 620.8 KB
Schema version  : 1.0
Run ID          : 2e32e596-6d90-4aea-ad4e-fb637f8b4696
Generated at    : 2026-10-14T19:10:54.500847+00:00
Accident periods: 84
Dev ages        : 84
Ultimate records: 84

No 'NaN' in JSON — OK
No 'Infinity' in JSON — OK
No '-Infinity' in JSON — OK


## Step 10 — Inspect the JSON document

**Purpose:** Demonstrate the JSON structure that the HTML viewer consumes.  
**Uses:** `doc` from Step 9  
**Produces:** Printed summaries of each top-level section  
**How it works:** Opening the JSON file in a text editor or running `json.loads` shows the
full structure. The four main sections — `triangle`, `development`, `ultimate_records`,
`assumptions` — correspond to the four tabs in the HTML viewer. Each section is self-contained:
the viewer can render any tab independently without cross-referencing the others.

In [10]:
# Top-level keys
print('Top-level keys:', list(doc.keys()))
print()
print('Triangle keys :', list(doc['triangle'].keys()))
print('Development keys:', list(doc['development'].keys()))
print()
print('First ultimate record:')
import pprint
pprint.pprint(doc['ultimate_records'][0])
print()
print('Last ultimate record:')
pprint.pprint(doc['ultimate_records'][-1])
print()
print('Assumptions section:')
pprint.pprint(doc['assumptions'])

Top-level keys: ['schema_version', 'run_id', 'generated_at', 'metadata', 'assumptions', 'triangle', 'development', 'ultimate_records']

Triangle keys : ['accident_periods', 'development_ages', 'cumulative', 'incremental', 'observed_mask']
Development keys: ['ldf_triangle', 'ata_matrix', 'atu_matrix']

First ultimate record:
{'accident_period': '2016-01',
 'actual_development_period': 'dev_83',
 'additional_development': 0.0,
 'applied_atu_period': None,
 'atu': 1.0,
 'current_incurred': 268376.2534695262,
 'flags': {'atu_below_1': False,
           'percent_developed_above_1': False,
           'ultimate_below_actual': False},
 'percent_developed': 1.0,
 'ultimate': 268376.2534695262}

Last ultimate record:
{'accident_period': '2022-12',
 'actual_development_period': 'dev_0',
 'additional_development': 580465.4593785625,
 'applied_atu_period': 'dev_1',
 'atu': 6.955658564288101,
 'current_incurred': 97464.5294240348,
 'flags': {'atu_below_1': False,
           'percent_developed_above_

## Step 11 — Export to Excel

**Purpose:** Write the teaching reference workbook with all six sheets.  
**Uses:** `export_to_excel(result, xlsx_path)`  
**Produces:** `teaching_development_method_2022-12.xlsx`  
**How it works:** `xlsxwriter` writes formula strings — it does not evaluate them. The
Formula Calculations sheet contains formulas like `=B5*C5` (Actual × ATU = Ultimate check) but the
cached numeric values in the file match the Python results at write time. Opening the workbook
in Excel and pressing Ctrl+Alt+F9 forces recalculation; column E should then match the Python
Reference sheet's Ultimate column. Changing an ATU value in column C updates columns E, F,
and G automatically through the formula chain. To experiment with formula changes, save a
copy of the workbook under a new name before editing.

In [11]:
xlsx_path = OUT_DIR / f'teaching_development_method_{CUTOFF}.xlsx'

export_to_excel(result, xlsx_path)
print(f'Workbook : {xlsx_path.name} — {xlsx_path.stat().st_size/1024:.1f} KB')

# Confirm sheets via openpyxl
import openpyxl
wb = openpyxl.load_workbook(xlsx_path)
print(f'Sheets   : {wb.sheetnames}')
wb.close()

Workbook : teaching_development_method_2022-12.xlsx — 146.6 KB
Sheets   : ['README', 'Cumulative', 'Incremental', 'Factors', 'Python Reference', 'Formula Calculations']


## Step 12 — Open the HTML viewer

**Purpose:** Load the precomputed JSON into the offline HTML viewer.  
**Uses:** `data/output/teaching_phase1/teaching_viewer.html`, the JSON from Step 9  
**Produces:** Interactive browser view with four tabs  
**How it works:** The HTML viewer uses the FileReader API — no server, no CORS, no network.
To open it:
1. Open `teaching_viewer.html` in any modern browser (Chrome, Firefox, Edge).
2. Click "Load JSON file" and select `teaching_export_2022-12.json`.
3. Navigate the four tabs: Triangle | Factors (ATA) | ATU / % Developed | Ultimates.
4. Use the Cumulative/Incremental toggle on the Triangle tab to switch views.
5. The Ultimates tab shows a flag summary if any diagnostic flags are set.

The viewer validates the schema version before rendering. An incompatible JSON will produce
an error message, not a broken display.  
**Windows note:** Open the HTML file directly from the filesystem (file:// protocol).
Do not use the viewer with a local web server unless CORS headers are set correctly.

In [12]:
viewer_path = OUT_DIR / 'teaching_viewer.html'
print(f'Viewer : {viewer_path}')
print(f'JSON   : {json_path}')
print()
print('To open:')
print(f'  1. Open {viewer_path} in a browser.')
print(f'  2. Click "Load JSON file" → select {json_path.name}')
print('  3. Explore all four tabs.')

Viewer : /mnt/data/Linux/Documents/IBNR_Utils/data/output/teaching_phase1/teaching_viewer.html
JSON   : /mnt/data/Linux/Documents/IBNR_Utils/data/output/teaching_phase1/teaching_export_2022-12.json

To open:
  1. Open /mnt/data/Linux/Documents/IBNR_Utils/data/output/teaching_phase1/teaching_viewer.html in a browser.
  2. Click "Load JSON file" → select teaching_export_2022-12.json
  3. Explore all four tabs.


## Step 13 — Reconciliation check

**Purpose:** Confirm that JSON ultimate values match direct function calculations.  
**Uses:** `doc`, `ult` from Step 7  
**Produces:** Max absolute difference (must be 0.0)  
**How it works:** This check demonstrates the audit trail: the JSON is not a standalone
output but a serialisation of the same numbers produced by `ultimate_matrix`. If anything
in the pipeline is modified, re-running this cell confirms whether the JSON is still consistent.

In [13]:
ult_from_json = [r['ultimate'] for r in doc['ultimate_records'] if r['ultimate'] is not None]
ult_direct    = result.ultimate_table['Ultimate'].dropna().tolist()

assert len(ult_from_json) == len(ult_direct), 'Record count mismatch'
diffs = [abs(a - b) for a, b in zip(ult_from_json, ult_direct)]
print(f'Max absolute diff JSON vs direct: {max(diffs):.2e}')
print(f'Total ultimate (JSON)    : {sum(ult_from_json):>18,.2f}')
print(f'Total ultimate (direct)  : {sum(ult_direct):>18,.2f}')
print()
print('Reconciliation: OK' if max(diffs) == 0 else 'WARNING: nonzero diff')

Max absolute diff JSON vs direct: 0.00e+00
Total ultimate (JSON)    :      36,934,740.16
Total ultimate (direct)  :      36,934,740.16

Reconciliation: OK
